# 26 — Trie Autocomplete (Amazon FAR style)

A prefix tree powering search autocomplete. Parts 1-3 build the core (concurrency at Part 3); Parts
4-5 are stretch tasks (deletion, then a parallel frequency precompute). Fill stubs, run each test
cell, peek at the collapsed `ref_` solutions only after trying.

A node is a dict of child chars; the marker key `"$"` holds the **insert count** of the word ending
there (so the same word inserted twice has count 2).

---

## Part 1 — Trie

`Trie` with `insert(word)`, `search(word) -> bool` (exact word present), and `starts_with(prefix) ->
bool` (any word has this prefix).

**Lock down:** `search` requires the end marker `"$"`; `starts_with` only needs the path to exist;
inserting the same word again bumps its count.

In [ ]:
class Trie:
    def __init__(self):
        raise NotImplementedError

    def insert(self, word):
        raise NotImplementedError

    def search(self, word):
        raise NotImplementedError

    def starts_with(self, prefix):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    t = Trie()
    for w in ("cat", "car", "card", "dog"):
        t.insert(w)
    assert t.search("car") and t.search("card") and not t.search("ca")
    assert t.starts_with("ca") and t.starts_with("do") and not t.starts_with("z")
    print("Part 1 OK")

_t1()

## Part 2 — Autocomplete (top-k by frequency)

`autocomplete(trie, prefix, k) -> list[word]`: every word under `prefix`, ranked by insert frequency
(descending), ties broken alphabetically, top `k`. Unknown prefix → `[]`.

**Lock down:** walk to the prefix node, DFS to collect `(word, count)`, sort by `(-count, word)`, slice
to `k`. This is the core ranked-suggestion query.

In [ ]:
def autocomplete(trie, prefix, k):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    t = Trie()
    for w in ("cat", "cat", "car", "card", "care"):
        t.insert(w)
    assert autocomplete(t, "ca", 2) == ["cat", "car"]   # cat (freq 2), then car (alpha tie)
    assert autocomplete(t, "car", 10) == ["car", "card", "care"]
    assert autocomplete(t, "z", 3) == []
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe trie

`ConcurrentTrie`: thread-safe `insert`/`search`. Many threads insert disjoint words at once; afterwards
every word is found.

**Discuss:** `setdefault` walks mutate shared nested dicts (resize races), so insert needs the lock;
a reader/writer lock since search dominates; lock-per-subtree for finer granularity.

In [ ]:
import threading


class ConcurrentTrie:
    def __init__(self):
        raise NotImplementedError

    def insert(self, word):
        raise NotImplementedError

    def search(self, word):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    t = ConcurrentTrie()

    def worker(b):
        for i in range(100):
            t.insert("w%d_%d" % (b, i))

    ts = [threading.Thread(target=worker, args=(b,)) for b in range(8)]
    for x in ts: x.start()
    for x in ts: x.join()
    assert all(t.search("w%d_%d" % (b, i)) for b in range(8) for i in range(100))
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Deletion

`DeletableTrie` extends the trie with `delete(word) -> bool`: decrement the word's count; once it hits
0 the word is no longer found. Returns `False` if the word isn't present (or already fully removed),
and a never-inserted word is a safe no-op.

**Discuss:** handling multiplicity (count, not a boolean flag), whether to prune now-empty nodes
(memory vs simplicity), and that deletion mustn't break sibling words sharing the path.

In [ ]:
class DeletableTrie(Trie):
    def delete(self, word):
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    t = DeletableTrie()
    t.insert("cat"); t.insert("cat"); t.insert("car")
    assert t.delete("cat") is True and t.search("cat") is True    # one "cat" left (inserted twice)
    assert t.delete("cat") is True and t.search("cat") is False   # now gone
    assert t.delete("cat") is False                               # already removed
    assert t.delete("zzz") is False                               # never inserted
    assert t.search("car") is True                                # sibling intact
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel frequency precompute

`parallel_counts(words, num_procs) -> dict[word, freq]`: count a large word list across processes with
`ProcessPoolExecutor`, merging the partial counts. These frequencies are the **insertion weights** you
then bulk-load into the trie for ranking. Worker is `trie_workers.count_words`.

**Discuss:** counting is a map-reduce (per-chunk count, then merge); GIL (CPU-bound -> processes); then
one bulk insert of `word × freq` builds the ranked trie.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import trie_workers


def parallel_counts(words, num_procs) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    words = ["a", "b", "a", "a", "c", "b"]
    assert parallel_counts(words, 2) == {"a": 3, "b": 2, "c": 1}
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Cache top-k completions at each prefix node for O(k) suggestions; update on insert.
- Fuzzy autocomplete (edit-distance-1) and typo tolerance.
- Radix/Patricia trie to compress chains; memory-efficient large vocabularies.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
from concurrent.futures import ProcessPoolExecutor
import trie_workers


class RefTrie:
    def __init__(self):
        self.root = {}

    def insert(self, word):
        node = self.root
        for c in word:
            node = node.setdefault(c, {})
        node["$"] = node.get("$", 0) + 1

    def search(self, word):
        node = self.root
        for c in word:
            if c not in node:
                return False
            node = node[c]
        return "$" in node

    def starts_with(self, prefix):
        node = self.root
        for c in prefix:
            if c not in node:
                return False
            node = node[c]
        return True


def ref_autocomplete(trie, prefix, k):
    node = trie.root
    for c in prefix:
        if c not in node:
            return []
        node = node[c]
    words = []

    def rec(n, path):
        if "$" in n:
            words.append((path, n["$"]))
        for c in sorted(ch for ch in n if ch != "$"):
            rec(n[c], path + c)

    rec(node, prefix)
    words.sort(key=lambda wc: (-wc[1], wc[0]))
    return [w for w, _ in words[:k]]


class RefConcurrentTrie(RefTrie):
    def __init__(self):
        super().__init__()
        self.lock = threading.Lock()

    def insert(self, word):
        with self.lock:
            super().insert(word)

    def search(self, word):
        with self.lock:
            return super().search(word)


class RefDeletableTrie(RefTrie):
    def delete(self, word):
        node = self.root
        for c in word:
            if c not in node:
                return False
            node = node[c]
        if "$" not in node:
            return False
        node["$"] -= 1
        if node["$"] == 0:
            del node["$"]
        return True


def ref_parallel_counts(words, num_procs):
    chunks = [words[i::num_procs] for i in range(num_procs)]
    total = {}
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        for d in ex.map(trie_workers.count_words, chunks):
            for w, c in d.items():
                total[w] = total.get(w, 0) + c
    return total


_t = RefTrie()
for _w in ("ab", "abc", "abc", "xy"):
    _t.insert(_w)
assert _t.search("abc") and not _t.search("a") and _t.starts_with("ab")
assert ref_autocomplete(_t, "ab", 2) == ["abc", "ab"]      # abc freq 2 first
_d = RefDeletableTrie(); _d.insert("a"); _d.insert("a")
assert _d.delete("a") and _d.search("a") and _d.delete("a") and not _d.search("a")
assert ref_parallel_counts(["p", "q", "p"], 2) == {"p": 2, "q": 1}
print("reference solutions OK")